In [8]:
import pymorphy2
import re
import csv
from collections import defaultdict
import os

# Инициализируем морфологический анализатор
morph = pymorphy2.MorphAnalyzer()

# Стоп-слова, которые не несут смысловой нагрузки
STOP_WORDS = {
    'в', 'на', 'с', 'и', 'а', 'но', 'или', 'у', 'к', 'о', 'от', 'из', 'для',
    'очень', 'сильно', 'немного', 'постоянно', 'иногда', 'периодически',
    'сегодня', 'вчера', 'утром', 'вечером', 'днем', 'ночью', 'это', 'что',
    'меня', 'у меня', 'уменя', 'есть', 'бывает', 'было', 'стало'
}

def load_knowledge_base_from_csv():
    """
    Загружает базу знаний из CSV файла
    
    Ожидаемые колонки: 'doctor' и 'symptoms'
    """
    knowledge_base = []
    filename='/home/jupyter-chirkovayd/dataset.csv'
    
    # Проверяем существование файла
    if not os.path.exists(filename):
        print(f"\n❌ Файл {filename} не найден!")
        # print("\nСоздаю пример файла doctors.csv с тестовыми данными...")
        # create_sample_csv(filename)
        # print(f"✅ Файл {filename} создан. Заполните его своими данными.")
        return knowledge_base
    
    try:
        with open(filename, 'r', encoding='utf-8-sig') as file:
            reader = csv.DictReader(file)
            
            # Проверяем наличие нужных колонок
            if 'doctor' not in reader.fieldnames or 'symptoms' not in reader.fieldnames:
                print(f"\n❌ Ошибка: в CSV файле отсутствуют нужные колонки!")
                print(f"Найдены колонки: {reader.fieldnames}")
                print("Необходимы колонки: 'doctor' и 'symptoms'")
                return knowledge_base
            
            # Загружаем данные
            for row_num, row in enumerate(reader, start=2):
                doctor = row.get('doctor', '').strip()
                symptoms = row.get('symptoms', '').strip()
                
                if doctor and symptoms:  # Пропускаем пустые строки
                    knowledge_base.append({
                        'doctor': doctor,
                        'symptoms': symptoms
                    })
                else:
                    print(f"⚠️ Предупреждение: строка {row_num} пропущена (пустые поля)")
            
            print(f"\n✅ Загружено {len(knowledge_base)} записей из файла {filename}")
            
            # Показываем первые несколько записей для проверки
            if knowledge_base:
                print("\nПервые 3 записи из базы:")
                for i, record in enumerate(knowledge_base[:3]):
                    print(f"  {i+1}. {record['doctor']}: {record['symptoms'][:50]}...")
            
    except Exception as e:
        print(f"\n❌ Ошибка при чтении файла: {e}")
    
    return knowledge_base



def normalize_word(word):
    """
    Приводит отдельное слово к нормальной форме (лемме)
    """
    # Очищаем от лишних символов
    word = re.sub(r'[^\w\-]', '', word.lower().strip())
    
    if not word or len(word) < 2 or word in STOP_WORDS:
        return None
    
    try:
        parsed = morph.parse(word)[0]
        return parsed.normal_form
    except:
        return word

def split_into_words(text):
    """
    Разбивает текст на отдельные слова
    """
    # Разделяем по пробелам и знакам препинания
    words = re.findall(r'[а-яёa-z\-]+', text.lower())
    return words

def symptoms_to_normalized_words(symptoms_string):
    """
    Преобразует строку симптомов в множество нормализованных слов
    """
    words = split_into_words(symptoms_string)
    normalized = set()
    
    for word in words:
        norm_word = normalize_word(word)
        if norm_word:
            normalized.add(norm_word)
    
    return normalized

def get_user_symptoms():
    """
    Получает симптомы от пользователя и разбивает на слова
    """
    print("\n" + "="*60)
    print("ОПИШИТЕ СВОИ СИМПТОМЫ")
    print("="*60)
    print("Введите ваши симптомы через запятую или пробел")
    print("Например: кашель, температура, болит голова, слабость")
    print("-"*60)
    
    user_input = input("Ваши симптомы: ").strip()
    
    if not user_input:
        print("⚠️ Симптомы не введены. Использую пример для демонстрации.")
        user_input = "кашлем, температура, голова болит, слабость"
    
    # Разбиваем на слова и нормализуем
    words = split_into_words(user_input)
    normalized_symptoms = set()
    
    print("\n📝 Разбор симптомов:")
    for word in words:
        norm_word = normalize_word(word)
        if norm_word:
            normalized_symptoms.add(norm_word)
            print(f"  ✅ '{word}' -> '{norm_word}'")
        else:
            print(f"  ⚠️ '{word}' -> (игнорируется)")
    
    return normalized_symptoms

def find_top_doctors(user_symptoms_set, knowledge_base, top_n=3):
    """
    Находит топ-N врачей на основе нормализованных слов симптомов
    """
    doctor_ratings = defaultdict(float)
    
    if not knowledge_base:
        print("\n❌ База знаний пуста!")
        return []
    
    print(f"\n🔄 Обработка базы знаний ({len(knowledge_base)} записей)...")
    
    for record in knowledge_base:
        # Нормализуем симптомы врача
        doctor_symptoms_set = symptoms_to_normalized_words(record['symptoms'])
        
        # Считаем пересечение (общие слова)
        common_symptoms = user_symptoms_set.intersection(doctor_symptoms_set)
        
        if len(doctor_symptoms_set) > 0:
            # Процент совпадения: (количество общих слов) / (количество слов врача)
            coverage_score = len(common_symptoms) / len(doctor_symptoms_set)
            
            # Процент от симптомов пользователя: (количество общих слов) / (количество слов пользователя)
            relevance_score = len(common_symptoms) / len(user_symptoms_set) if user_symptoms_set else 0
            
            # Комбинированный рейтинг
            final_score = coverage_score * 0.4 + relevance_score * 0.6
        else:
            final_score = 0
        
        # Разделяем врачей, если их несколько
        doctors = [d.strip().lower() for d in record['doctor'].split(',')]
        
        for doctor in doctors:
            if final_score > doctor_ratings[doctor]:
                doctor_ratings[doctor] = final_score
    
    # Сортируем врачей по убыванию рейтинга
    sorted_doctors = sorted(doctor_ratings.items(), key=lambda x: x[1], reverse=True)
    
    return sorted_doctors[:top_n]

def format_results(user_symptoms_set, sorted_doctors, knowledge_base):
    """
    Форматирует результаты для вывода
    """
    print("\n" + "="*60)
    print("РЕЗУЛЬТАТЫ ПОИСКА")
    print("="*60)
    print(f"✅ Ваши симптомы (нормализованные): {', '.join(sorted(user_symptoms_set))}")
    print("-"*60)
    
    if not sorted_doctors:
        print("❌ К сожалению, не удалось найти подходящих врачей.")
        print("💡 Попробуйте описать симптомы подробнее или другими словами.")
        return
    
    print("\n🏆 ТОП-3 РЕКОМЕНДУЕМЫХ ВРАЧА:")
    print()
    
    for i, (doctor, score) in enumerate(sorted_doctors, 1):
        percentage = score * 100
        
        # Определяем уровень совпадения
        if percentage >= 30:
            level = "🔴 ВЫСОКАЯ ВЕРОЯТНОСТЬ"
        elif percentage >= 15:
            level = "🟡 СРЕДНЯЯ ВЕРОЯТНОСТЬ"
        else:
            level = "🟢 НИЗКАЯ ВЕРОЯТНОСТЬ"
        
        # Находим этого врача в базе, чтобы показать его симптомы
        doctor_info = None
        for record in knowledge_base:
            if doctor in [d.strip().lower() for d in record['doctor'].split(',')]:
                doctor_info = record
                break
        
        print(f"{i}. 👨‍⚕️ {doctor.title()}")
        print(f"   {level}")
        print(f"   📊 Совпадение: {percentage:.1f}%")
        if doctor_info:
            # Показываем только первые 100 символов симптомов
            symptoms_preview = doctor_info['symptoms'][:100] + "..." if len(doctor_info['symptoms']) > 100 else doctor_info['symptoms']
            print(f"   📋 Лечит: {symptoms_preview}")
        print()

def main():
    """
    Главная функция программы
    """
    print("="*60)
    print("🏥 ПОМОЩНИК ВЫБОРА ВРАЧА")
    print("="*60)
    print("Программа поможет определить, к какому врачу лучше записаться")
    print("на основе описанных вами симптомов.")
    
    # Загружаем базу знаний из CSV
    # csv_filename = input("\n📁 Введите имя CSV файла (Enter для 'doctors.csv'): ").strip()
    # if not csv_filename:
    #     csv_filename = 'doctors.csv'
    
    knowledge_base = load_knowledge_base_from_csv()
    
    if not knowledge_base:
        print("\n❌ Не удалось загрузить базу знаний. Программа завершена.")
        print("💡 Убедитесь, что файл существует и содержит колонки 'doctor' и 'symptoms'")
        return
    
    while True:
        # Получаем симптомы пользователя
        user_symptoms_set = get_user_symptoms()
        
        if not user_symptoms_set:
            print("⚠️ Не удалось распознать симптомы. Попробуйте еще раз.")
            continue
        
        # Ищем врачей
        top_doctors = find_top_doctors(user_symptoms_set, knowledge_base, top_n=3)
        
        # Выводим результаты
        format_results(user_symptoms_set, top_doctors, knowledge_base)
        
        # Спрашиваем, хочет ли пользователь продолжить
        print("\n" + "-"*60)
        choice = input("❓ Хотите ввести другие симптомы? (да/нет): ").strip().lower()
        if choice not in ['да', 'lf', 'yes', 'y', 'д', 'да ', 'yes ']:
            print("\n✅ Спасибо за использование сервиса! Выздоравливайте! 🏥")
            break

if __name__ == "__main__":
    # Проверяем установку pymorphy2
    try:
        import pymorphy2
    except ImportError:
        print("📦 Устанавливаем необходимые библиотеки...")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pymorphy2"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pymorphy2-dicts-ru"])
        print("✅ Библиотеки успешно установлены!")
        import pymorphy2
    
    main()

🏥 ПОМОЩНИК ВЫБОРА ВРАЧА
Программа поможет определить, к какому врачу лучше записаться
на основе описанных вами симптомов.

✅ Загружено 99 записей из файла /home/jupyter-chirkovayd/dataset.csv

Первые 3 записи из базы:
  1. family doctor,urgent care: fever,cough,sore throat,runny or stuffy nose,muscl...
  2. family doctor,pulmonologist: cough,mucus production,shortness of breath,chest p...
  3. family doctor,pulmonologist: fever,cough,shortness of breath,chest pain,fatigue...

ОПИШИТЕ СВОИ СИМПТОМЫ
Введите ваши симптомы через запятую или пробел
Например: кашель, температура, болит голова, слабость
------------------------------------------------------------


Ваши симптомы:  acute pain in the abdomen on the right



📝 Разбор симптомов:
  ✅ 'acute' -> 'acute'
  ✅ 'pain' -> 'pain'
  ✅ 'in' -> 'in'
  ✅ 'the' -> 'the'
  ✅ 'abdomen' -> 'abdomen'
  ✅ 'on' -> 'on'
  ✅ 'the' -> 'the'
  ✅ 'right' -> 'right'

🔄 Обработка базы знаний (99 записей)...

РЕЗУЛЬТАТЫ ПОИСКА
✅ Ваши симптомы (нормализованные): abdomen, acute, in, on, pain, right, the
------------------------------------------------------------

🏆 ТОП-3 РЕКОМЕНДУЕМЫХ ВРАЧА:

1. 👨‍⚕️ Surgeon
   🔴 ВЫСОКАЯ ВЕРОЯТНОСТЬ
   📊 Совпадение: 65.1%
   📋 Лечит: pain in the lower right abdomen,nausea,vomiting,fever

2. 👨‍⚕️ Gastroenterologist
   🔴 ВЫСОКАЯ ВЕРОЯТНОСТЬ
   📊 Совпадение: 54.3%
   📋 Лечит: vomiting,diarrhea,jaundice,liver damage,death

3. 👨‍⚕️ Rheumatologist
   🔴 ВЫСОКАЯ ВЕРОЯТНОСТЬ
   📊 Совпадение: 45.7%
   📋 Лечит: pain,stiffness,swelling,inflammation in the joints


------------------------------------------------------------


❓ Хотите ввести другие симптомы? (да/нет):  нет



✅ Спасибо за использование сервиса! Выздоравливайте! 🏥
